In [ ]:
from function_lib import *

In [ ]:


class SPOParam:
    """
    Nonconvex SPO dual formulation with Gurobi.

    Variables:
      B[k,j]      : d_x × d_c matrix
      y[n,j]      : predicted cost (B^T x_n)_j  (auxiliary)
      mu[n,i]     : ≥ 0
      nu[n,i]     : ≥ 0
      pi[n,j]     : free
      rho[n,j]    : free
      lam[n]      : free
      eta[n]      : free

    Objective (matches your LaTeX, per n):
      sum_n [ lam_n + eta_n
              + sum_i mu_{n,i} (B^T x_n)^T z_i
              + sum_i nu_{n,i} c_n^T z_i ]

    Constraints (per n, component-wise in j):
      c_n = pi_n + (sum_i mu_{n,i}) (B^T x_n)
      -c_n = rho_n + (sum_i nu_{n,i}) c_n
      0 <= -pi_n^T z_i + lam_n        ∀i
      0 <= -rho_n^T z_i + eta_n       ∀i

    Model is built once; X and C only change coefficients (and RHS) across runs.
    """

    def __init__(self, N, d_x, d_c, Z):
        self.N = int(N)
        self.d_x = int(d_x)
        self.d_c = int(d_c)
        self.Z = np.asarray(Z, dtype=float)    # (K, d_c)
        self.K, d_c2 = self.Z.shape
        if d_c2 != self.d_c:
            raise ValueError("Z must have shape (K, d_c) with d_c matching d_c.")

        # Build Gurobi model once
        self._build_model()

    # --------------------------------------------------------
    # Build model (no X, C yet – these enter as coeffs/RHS)
    # --------------------------------------------------------
    def _build_model(self):
        N, d_x, d_c, K = self.N, self.d_x, self.d_c, self.K
        Z = self.Z

        m = gp.Model("spo_param")
        self.model = m

        # Allow nonconvex quadratic
        m.Params.NonConvex = 2

        # ----------------------------------------------------
        # Variables
        # ----------------------------------------------------
        self.B   = m.addVars(d_x, d_c, lb=-GRB.INFINITY, name="B")
        self.y   = m.addVars(N,   d_c, lb=-GRB.INFINITY, name="y")     # y[n,j] = (B^T x_n)_j

        self.mu  = m.addVars(N, K, lb=0.0,           name="mu")
        self.nu  = m.addVars(N, K, lb=0.0,           name="nu")
        self.pi  = m.addVars(N, d_c, lb=-GRB.INFINITY, name="pi")
        self.rho = m.addVars(N, d_c, lb=-GRB.INFINITY, name="rho")
        self.lam = m.addVars(N, lb=-GRB.INFINITY,    name="lam")
        self.eta = m.addVars(N, lb=-GRB.INFINITY,    name="eta")

        # ----------------------------------------------------
        # 1) y[n,j] = sum_k B[k,j] * X[n,k]
        # We don't yet know X, so we create empty constraints
        #    y[n,j] == 0
        # and later add/update B coefficients with chgCoeff()
        # ----------------------------------------------------
        self.constr_y = {}
        for n in range(N):
            for j in range(d_c):
                self.constr_y[(n, j)] = m.addConstr(self.y[n, j] == 0.0,
                                                    name=f"y_def[{n},{j}]")

        # ----------------------------------------------------
        # 2) c_n = pi_n + (sum_i mu_{n,i}) * y_n
        # Component-wise: pi[n,j] + sum_i mu[n,i] * y[n,j] = c[n,j]
        # This is bilinear (mu * y) → QConstr.
        # RHS initially 0; updated to c[n,j] later.
        # ----------------------------------------------------
        self.constr_ceq = {}
        for n in range(N):
            for j in range(d_c):
                qexpr = gp.QuadExpr()
                qexpr += self.pi[n, j]
                for i in range(K):
                    qexpr += self.mu[n, i] * self.y[n, j]
                self.constr_ceq[(n, j)] = m.addQConstr(qexpr == 0.0,
                                                        name=f"c_eq[{n},{j}]")

        # ----------------------------------------------------
        # 3) -c_n = rho_n + (sum_i nu_{n,i}) c_n
        # Component-wise: rho[n,j] + sum_i nu[n,i] * c[n,j] = - c[n,j]
        # This is linear in (rho, nu) but uses c[n,j] as coefficients.
        # We build with c=0 and update coefficients & RHS later.
        # ----------------------------------------------------
        self.constr_negc = {}
        for n in range(N):
            for j in range(d_c):
                # initially rho[n,j] == 0
                self.constr_negc[(n, j)] = m.addConstr(self.rho[n, j] == 0.0,
                                                        name=f"negc[{n},{j}]")

        # ----------------------------------------------------
        # 4) 0 <= -pi_n^T z_i + lam_n
        #    lam[n] - sum_j pi[n,j] * z[i,j] >= 0
        # ----------------------------------------------------
        for n in range(N):
            for i in range(K):
                expr = gp.LinExpr()
                expr += self.lam[n]
                for j in range(d_c):
                    if Z[i, j] != 0.0:
                        expr += -Z[i, j] * self.pi[n, j]
                m.addConstr(expr >= 0.0, name=f"pi_dual[{n},{i}]")

        # ----------------------------------------------------
        # 5) 0 <= -rho_n^T z_i + eta_n
        #    eta[n] - sum_j rho[n,j] * z[i,j] >= 0
        # ----------------------------------------------------
        for n in range(N):
            for i in range(K):
                expr = gp.LinExpr()
                expr += self.eta[n]
                for j in range(d_c):
                    if Z[i, j] != 0.0:
                        expr += -Z[i, j] * self.rho[n, j]
                m.addConstr(expr >= 0.0, name=f"rho_dual[{n},{i}]")

        # ----------------------------------------------------
        # Objective:
        #   sum_n [ lam_n + eta_n
        #           + sum_i mu_{n,i} y_n^T z_i
        #           + sum_i nu_{n,i} c_n^T z_i ]
        #
        # The nu-term is linear and depends on C; we set its
        # coefficients later via nu[n,i].Obj.
        # ----------------------------------------------------
        obj = gp.QuadExpr()

        # lam_n + eta_n terms (linear, constant across runs)
        for n in range(N):
            obj += self.lam[n]
            obj += self.eta[n]

        # mu * y bilinear terms with coefficient z[i,j]
        for n in range(N):
            for i in range(K):
                for j in range(d_c):
                    if Z[i, j] != 0.0:
                        obj += Z[i, j] * self.mu[n, i] * self.y[n, j]

        # We'll add nu-part via Obj coefficients later (depends on C)
        m.setObjective(obj, GRB.MINIMIZE)

        m.update()

    # --------------------------------------------------------
    # PUBLIC: solve for given dataset (X, C)
    #         update coefficients only, not rebuild.
    # --------------------------------------------------------
    def solve(self, X, C, verbose=False):
        """
        Solve the SPO dual for a given dataset (X, C).

        Parameters
        ----------
        X : array, shape (N, d_x)
            Feature matrix (rows = x_n^T).
        C : array, shape (N, d_c)
            Cost matrix (rows = c_n^T).
        verbose : bool
            If False, suppress Gurobi output.

        Returns
        -------
        result : dict with keys:
            "B"   : optimal B (d_x, d_c)
            "obj" : optimal objective value
            also the dual stuff if you want: mu, nu, pi, rho, lam, eta
        """
        X = np.asarray(X, dtype=float)
        C = np.asarray(C, dtype=float)

        if X.shape != (self.N, self.d_x):
            raise ValueError(f"X must have shape ({self.N}, {self.d_x})")
        if C.shape != (self.N, self.d_c):
            raise ValueError(f"C must have shape ({self.N}, {self.d_c})")

        N, d_x, d_c, K = self.N, self.d_x, self.d_c, self.K
        Z = self.Z
        m = self.model

        if not verbose:
            m.Params.OutputFlag = 0
        else:
            m.Params.OutputFlag = 1

        # ----------------------------------------------------
        # 1) Update y-definition constraints: y[n,j] = sum_k B[k,j] * X[n,k]
        #    We change the coefficients of B[k,j] in constr_y[(n,j)].
        # ----------------------------------------------------
        for n in range(N):
            for j in range(d_c):
                constr = self.constr_y[(n, j)]
                # First zero out old B-coeffs (or overwrite directly)
                for k in range(d_x):
                    m.chgCoeff(constr, self.B[k, j], -X[n, k])
                # RHS remains 0: y[n,j] - sum_k X[n,k] B[k,j] = 0

        # ----------------------------------------------------
        # 2) Update c_eq RHS: pi[n,j] + sum_i mu[n,i] y[n,j] = C[n,j]
        #    Only RHS depends on C.
        # ----------------------------------------------------
        for n in range(N):
            for j in range(d_c):
                qconstr = self.constr_ceq[(n, j)]
                qconstr.QCRHS = C[n, j]

        # ----------------------------------------------------
        # 3) Update -c_n = rho_n + sum_i nu_{n,i} c_n
        #    rho[n,j] + sum_i nu[n,i] * C[n,j] = -C[n,j]
        #    So linear coeffs for nu and RHS depend on C.
        # ----------------------------------------------------
        for n in range(N):
            for j in range(d_c):
                constr = self.constr_negc[(n, j)]
                # Update nu-coefficients
                for i in range(K):
                    m.chgCoeff(constr, self.nu[n, i], C[n, j])
                # Update RHS
                constr.RHS = -C[n, j]

        # ----------------------------------------------------
        # 4) Update objective linear coefficients for nu:
        #    sum_i nu_{n,i} c_n^T z_i
        #    coeff_{n,i} = c_n^T z_i
        # ----------------------------------------------------
        # First reset all nu obj parts to 0
        for n in range(N):
            for i in range(K):
                self.nu[n, i].Obj = 0.0

        # Now set the correct coefficients, depending on C and Z.
        for n in range(N):
            for i in range(K):
                coeff = float(C[n, :].dot(Z[i, :]))  # c_n^T z_i
                if coeff != 0.0:
                    self.nu[n, i].Obj = coeff

        m.update()

        # ----------------------------------------------------
        # Optimize
        # ----------------------------------------------------
        m.optimize()

        if m.status != GRB.OPTIMAL:
            raise RuntimeError(f"Gurobi did not find an optimal solution. Status: {m.status}")

        # Extract solution
        B_val = np.zeros((d_x, d_c))
        for k in range(d_x):
            for j in range(d_c):
                B_val[k, j] = self.B[k, j].X

        # Optional: extract dual variables if you care
        mu_val  = np.zeros((N, K))
        nu_val  = np.zeros((N, K))
        pi_val  = np.zeros((N, d_c))
        rho_val = np.zeros((N, d_c))
        lam_val = np.zeros(N)
        eta_val = np.zeros(N)

        for n in range(N):
            lam_val[n] = self.lam[n].X
            eta_val[n] = self.eta[n].X
            for i in range(K):
                mu_val[n, i] = self.mu[n, i].X
                nu_val[n, i] = self.nu[n, i].X
            for j in range(d_c):
                pi_val[n, j]  = self.pi[n, j].X
                rho_val[n, j] = self.rho[n, j].X

        obj_val = m.ObjVal

        return {
            "B":   B_val,
            "mu":  mu_val,
            "nu":  nu_val,
            "pi":  pi_val,
            "rho": rho_val,
            "lam": lam_val,
            "eta": eta_val,
            "obj": obj_val,
        }

    # --------------------------------------------------------
    # Same regret evaluation as your LSTECPParam
    # --------------------------------------------------------
    def evaluate_regret(self, X_test, C_test, B=None, return_per_sample=False):
        """
        Vectorized regret evaluation (same as your previous class).

        Parameters
        ----------
        X_test : array, shape (N_test, d_x)
        C_test : array, shape (N_test, d_c)
        B      : optional B matrix; if None, use current model B.

        Returns
        -------
        result : dict with "avg_regret" and optionally "regret_per_sample".
        """
        X_test = np.asarray(X_test, dtype=float)
        C_test = np.asarray(C_test, dtype=float)

        N_test, d_x_test = X_test.shape
        N_ctest, d_c_test = C_test.shape

        if d_x_test != self.d_x:
            raise ValueError(f"X_test shape mismatch: expected dim {self.d_x}, got {d_x_test}")
        if d_c_test != self.d_c:
            raise ValueError(f"C_test shape mismatch: expected dim {self.d_c}, got {d_c_test}")
        if N_test != N_ctest:
            raise ValueError("X_test and C_test must have the same number of rows.")

        if B is None:
            # Get B from current model
            B = np.zeros((self.d_x, self.d_c))
            for k in range(self.d_x):
                for j in range(self.d_c):
                    B[k, j] = self.B[k, j].X
        else:
            B = np.asarray(B, dtype=float).reshape(self.d_x, self.d_c)

        # Predicted costs
        C_hat = X_test @ B                    # (N_test, d_c)

        # Objective for all feasible z on predicted costs
        Z = self.Z                            # (K, d_c)
        obj_pred = C_hat @ Z.T                # (N_test, K)

        # Predicted decision index
        idx_hat = np.argmin(obj_pred, axis=1) # (N_test,)

        # Objective for all feasible z on true costs
        obj_true = C_test @ Z.T               # (N_test, K)

        # Costs of chosen vs optimal decisions
        chosen_costs  = obj_true[np.arange(N_test), idx_hat]
        optimal_costs = obj_true.min(axis=1)

        regrets    = chosen_costs - optimal_costs
        avg_regret = float(regrets.mean())

        result = {"avg_regret": avg_regret}
        if return_per_sample:
            result["regret_per_sample"] = regrets
        return result


In [ ]:
# ----------------- config -----------------
n_instance   = 1000
input_dim    = 5
noise        = 0.5
deg          = 2
grid_width   = 5
epoch        = 100
learning_rate = 0.001
n_simulation = 50

# ----------------- paths ------------------
base_path = change_to_py_file_dir()
base      = os.path.join(base_path, "Data", "Shortest_path")

# ----------------- results ----------------
total_result  = {}
# =========================================================
# main loop over seeds
# =========================================================

# ----- build Z (all feasible z) only once -----
sols, edges, G = generate_all_feasible_solutions(grid_width=5)

model_list = ["SPO"]

for model_type in model_list:

    regret_result = []
    parame_result = []
    
    solver = None  # will build once on the first iteration

    for seed in tqdm(range(n_simulation),
                     desc="Running simulations",
                     colour="green"):

        # ---- reproducibility ----
        random.seed(seed)
        np.random.seed(seed)

        # ---- data for this seed ----
        x_train, y_train, x_test, y_test = load_shortest_path_setting_with_splits(
            base, n_instance, input_dim, deg, noise, grid_width, seed
        )
        
        # ---- build solver only once (reuse the same model) ----
        if solver is None:
            solver = SPOParam(
                N=x_train.shape[0],       # must be constant across seeds
                d_x=x_train.shape[1],
                d_c=y_train.shape[1],
                Z=sols
            )

        # ---- solve training problem for this seed ----
        res = solver.solve(x_train, y_train, verbose=True)

        # ---- evaluate regret on test set ----
        # B is already stored in solver.B after solve(), so B argument is optional
        eval_res = solver.evaluate_regret(
            x_test, y_test, return_per_sample=False
        )

        regret_result.append(eval_res["avg_regret"])
        parame_result.append(res["B"])

    # save results for this model type
    total_result[model_type] = {
        "regret": regret_result,
        "params": parame_result
    }
    # ==================================================
    # CRITICAL FIX: Clean up memory before next model!
    # ==================================================
    if solver is not None:
        solver.M.dispose()  # Release Mosek C-memory immediately
        del solver          # Delete Python object
    
    gc.collect()            # Force Python Garbage Collector

    process = psutil.Process(os.getpid())
    print(f"Python process RAM: {process.memory_info().rss/1024/1024/1024:.3f} GB")

# Save the result to pickle file
save_path = os.path.join(base_path, "Results", "Score")
os.makedirs(save_path, exist_ok=True)   # create directory if not exist
file_path = os.path.join(save_path, f"Shortest_path_spo_result_{n_instance}_{input_dim}_{deg}_{noise}_{grid_width}.pkl")
with open(file_path, "wb") as f:
    pickle.dump(total_result, f)

def print_memory():
    mem = psutil.virtual_memory()
    print(f"Available: {mem.available/1024/1024/1024:.2f} GB")
    print(f"Used:      {(mem.total - mem.available)/1024/1024/1024:.2f} GB")
    print(f"Percent:   {mem.percent}%")

print_memory()

In [ ]:
# Compute stats for the LST model
# regrets = total_result["LST-0.1"]["regret"]

# mean, std, var90, cvar90 = compute_stats(regrets)

# print(f"Mean    : {mean:.6f}")
# print(f"Std     : {std:.6f}")
# print(f"VaR 90  : {var90:.6f}")
# print(f"CVaR 90 : {cvar90:.6f}")
rows = []
for model_type in model_list:
    regrets = total_result[model_type]["regret"]
    mean, std, var90, cvar90 = compute_stats(regrets)

    rows.append({
        "Model": model_type,
        "Mean": mean,
        "Std": std,
        "VaR90": var90,
        "CVaR90": cvar90
    })


# convert to DataFrame
df = pd.DataFrame(rows)

# nicely formatted print
print(df.to_string(index=False, float_format="%.6f"))


# Collect regret distributions across all model types
df_list = []
for model_type, data in total_result.items():
    df_list.append(pd.DataFrame({
        "method": [model_type] * len(data["regret"]),
        "regret": data["regret"]
    }))

df = pd.concat(df_list, ignore_index=True)

# Academic color palette
palette = sns.color_palette("deep")
unique_methods = sorted(df["method"].unique())
method_colors = {m: palette[i % len(palette)] for i, m in enumerate(unique_methods)}

plt.figure(figsize=(6, 4))

sns.boxplot(
    data=df,
    x="method",
    y="regret",
    palette=method_colors,
    showmeans=True,
    meanprops={
        "marker": "D",
        "markerfacecolor": "white",
        "markeredgecolor": "black",
        "markersize": 5
    }
)

sns.stripplot(
    data=df,
    x="method",
    y="regret",
    color="black",
    alpha=0.25,
    jitter=0.2,
    size=3
)

plt.title("Regret Distribution Across Methods", fontsize=12)
plt.ylabel("Regret")
plt.xlabel("Method")
plt.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()

# Save figure
save_dir = os.path.join(base_path, "Results", "Figures")
os.makedirs(save_dir, exist_ok=True)

plt.savefig(
    os.path.join(save_dir, "regret_comparison.png"),
    dpi=400,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# seed = 1
# x_train, y_train, x_test, y_test = load_shortest_path_setting_with_splits(base, n_instance, input_dim, deg, noise, grid_width, seed)

# # Example usage:

# sols, edges, G = generate_all_feasible_solutions(grid_width=5)
# print("Number of edges:", len(edges))
# print("Number of feasible s->t paths:", len(sols))
# print("First solution vector:", sols[0])

# # pick any solution
# vec = sols[10]
# # draw
# draw_solution(G, edges, vec, grid_width=5, cost=y_train[0])
# solver = LSTECPParam(
#     N=x_train.shape[0],
#     d_x=x_train.shape[1],
#     d_c=y_train.shape[1],
#     Z=sols,
#     eta=0.5
# )

# X_run = x_train   # shape (N, d_x)
# C_run = y_train   # shape (N, d_c)

# res = solver.solve(X_run, C_run, verbose=False)
# eval_res = solver.evaluate_regret(x_test, y_test, return_per_sample=True)
# print("Avg regret:", eval_res["avg_regret"])
# # print("Per-sample regret:", eval_res["regret_per_sample"])
